# Actor-critic PPO, derived and smoke-tested

The counterpart to [`../../membrane_grpo/grpo_scratch.py`](../../membrane_grpo/grpo_scratch.py).
Same task, same frozen data, same deterministic reward, same 0.5B model, same clipped
surrogate. One thing moves: **where the baseline comes from, and how finely credit is
assigned.**

| | GRPO | actor-critic PPO (here) |
| --- | --- | --- |
| baseline | mean reward of $G$ samples of *one* prompt | learned $V_\phi(s_t)$ |
| advantage | one scalar per sequence, broadcast to every token | one per **token**, via GAE |
| rollouts per prompt | $G$ (8 in the reference run) | 1 is enough |
| degenerate case | all $G$ rewards equal $\Rightarrow$ advantages exactly $0$, no gradient | none |
| extra parameters | none | $d_{\text{model}} + 1 = 897$ |

This notebook is the **derivation and the smoke tests**: everything that can be checked
without a GPU, checked. It runs on CPU in about a minute.
[`02_ppo_actor_critic_0.5b.ipynb`](02_ppo_actor_critic_0.5b.ipynb) is the run itself.

Every check below is an `assert`. If this notebook executes to the end, the claims in it
are true of the code as committed.

In [1]:
import sys, os, json
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))

import torch
import ppo_ac
from ppo_ac import (
    gae, terminal_rewards, ppo_actor_loss, value_loss, whiten,
    ValueHead, padded_position_ids, VALUE_INIT_BIAS,
)

torch.manual_seed(0)
print("torch", torch.__version__, "| reward + data from", ppo_ac.DATA.parent.name,
      "| prompt", ppo_ac.PROMPT_VERSION)

torch 2.13.0+cu130 | reward + data from membrane_grpo | prompt v2


## 1. The task is a bandit; PPO wants a trajectory

One prompt, one completion, one terminal scalar. To use PPO the completion is read as a
trajectory over tokens:

$$
s_t = (\text{prompt},\, a_0 \dots a_{t-1}), \qquad
r_t = \begin{cases} 0 & t < L-1 \\ R & t = L-1 \end{cases}, \qquad \gamma = 1
$$

$L$ is the **active** length: up to and including the first `EOS`. Deciding to stop is an
action, so it is scored; padding was never sampled, so it is not. That mask comes from
`grpo_scratch.build_mask`, imported rather than reimplemented — one tokenizer convention
across both methods, or the comparison is between two different tasks.

$\gamma = 1$ because completions are ~110 tokens and nothing about this task justifies
preferring an early token to a late one.

In [2]:
B, T = 4, 7
lengths = [7, 5, 3, 1]
mask = torch.zeros(B, T)
for i, L in enumerate(lengths):
    mask[i, :L] = 1.0

values = torch.randn(B, T)
R = torch.tensor([0.9, 0.0, 0.35, 0.5])
rewards = terminal_rewards(R, mask)

# CHECK 1 -- the reward lands on the last active token of its own row, and nowhere else.
for i, L in enumerate(lengths):
    assert rewards[i, L - 1] == R[i]
    assert rewards[i].abs().sum() == abs(R[i])
    assert rewards[i, L:].abs().sum() == 0
print("CHECK 1 ok  | row 2 (L=3):", rewards[2].tolist())

CHECK 1 ok  | row 2 (L=3): [0.0, 0.0, 0.3499999940395355, 0.0, 0.0, 0.0, 0.0]


## 2. Why a baseline at all

The policy gradient is unchanged by subtracting any $b(s_t)$ that does not depend on the
action:

$$
\nabla_\theta J = \mathbb{E}\Big[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t)\,
\big(G_t - b(s_t)\big)\Big], \qquad
\mathbb{E}\big[\nabla \log \pi \cdot b(s_t)\big] = b(s_t)\,\nabla \textstyle\sum_a \pi = 0 .
$$

So the baseline is free in expectation and pure variance reduction in practice. The two
methods differ only in what they put in $b$:

* **GRPO** — $b = \frac{1}{G}\sum_i R_i$, the mean over $G$ samples *of the same prompt*.
  Costs $G$ rollouts, needs no parameters, and is exactly right by construction.
* **Actor-critic** — $b = V_\phi(s_t)$, a learned function. Costs one linear head, needs
  one rollout, and can be *wrong* — which is the trade this experiment is about.

## 3. GAE

With $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and $V(s_L) = 0$ at the terminal:

$$
A_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l \ge 0} (\gamma\lambda)^l \delta_{t+l}
\qquad\Longleftrightarrow\qquad
A_t = \delta_t + \gamma\lambda A_{t+1}, \quad A_t = 0 \text{ for } t \ge L .
$$

Two limits are worth holding on to, because they bracket the whole bias-variance dial:

$$
\lambda = 1:\quad A_t = R - V(s_t) \qquad\text{(unbiased; all of the trajectory's noise)}
$$
$$
\lambda = 0:\quad A_t = V(s_{t+1}) - V(s_t) \quad (t < L-1) \qquad\text{(low variance; only as good as } V)
$$

The critic's regression target is the GAE return $G_t = A_t + V(s_t)$, which at
$\lambda = 1$ is simply $R$ at every position.

In [3]:
# CHECK 2 -- lambda = 1 collapses to R - V(s_t), exactly.
A1, G1 = gae(rewards, values, mask, gamma=1.0, lam=1.0)
want = (R.unsqueeze(-1) - values) * mask
assert torch.allclose(A1, want, atol=1e-6)
print(f"CHECK 2 ok  | max |A - (R-V)| = {float((A1-want).abs().max()):.2e}")

# CHECK 3 -- and its return is R on every active token.
assert torch.allclose(G1, R.unsqueeze(-1) * mask, atol=1e-6)
print("CHECK 3 ok  | returns at lam=1 are R everywhere active")

# CHECK 4 -- lambda = 0 collapses to the one-step TD residual.
A0, _ = gae(rewards, values, mask, gamma=1.0, lam=0.0)
for i, L in enumerate(lengths):
    for t in range(L - 1):
        assert abs(A0[i, t] - (values[i, t+1] - values[i, t])) < 1e-6
    assert abs(A0[i, L-1] - (R[i] - values[i, L-1])) < 1e-6
    assert A0[i, L:].abs().sum() == 0
print("CHECK 4 ok  | lam=0 is V(s_{t+1}) - V(s_t), and R - V at the terminal")

CHECK 2 ok  | max |A - (R-V)| = 5.96e-08
CHECK 3 ok  | returns at lam=1 are R everywhere active
CHECK 4 ok  | lam=0 is V(s_{t+1}) - V(s_t), and R - V at the terminal


In [4]:
# CHECK 5 -- the recursion equals the explicit discounted sum of deltas.
g, lam = 0.97, 0.8
A, _ = gae(rewards, values, mask, gamma=g, lam=lam)
for i, L in enumerate(lengths):
    d = []
    for t in range(L):
        nv = values[i, t+1] if t + 1 < L else torch.tensor(0.0)
        d.append(rewards[i, t] + g * nv - values[i, t])
    for t in range(L):
        explicit = sum(((g * lam) ** l) * d[t + l] for l in range(L - t))
        assert abs(A[i, t] - explicit) < 1e-5
print("CHECK 5 ok  | A_t = sum_l (gamma*lam)^l delta_{t+l}")

# CHECK 6 -- padding is inert. Garbage past the mask must not leak in, and the
# accumulator must not carry backwards across the boundary.
poisoned = values.clone()
for i, L in enumerate(lengths):
    poisoned[i, L:] = 999.0
A_p, _ = gae(rewards, poisoned, mask, gamma=g, lam=lam)
assert torch.allclose(A_p * mask, A * mask, atol=1e-5)
print(f"CHECK 6 ok  | V=999 past the mask changes nothing: {float(((A_p-A)*mask).abs().max()):.1e}")

CHECK 5 ok  | A_t = sum_l (gamma*lam)^l delta_{t+l}
CHECK 6 ok  | V=999 past the mask changes nothing: 0.0e+00


## 4. The clipped surrogate, and its gradient by hand

$$
\rho_t = \exp(\log \pi_\theta(a_t) - \log \pi_{\text{old}}(a_t)), \qquad
\mathcal{L}_t = \min\!\big(\rho_t A_t,\ \mathrm{clip}(\rho_t, 1{-}\epsilon, 1{+}\epsilon) A_t\big)
$$

Differentiating with respect to $\log \pi$, using $\partial \rho / \partial \log \pi = \rho$:

$$
\frac{\partial \mathcal{L}_t}{\partial \log \pi_\theta(a_t)} =
\begin{cases}
0 & A_t > 0 \ \text{and}\ \rho_t > 1+\epsilon \\
0 & A_t < 0 \ \text{and}\ \rho_t < 1-\epsilon \\
\rho_t A_t & \text{otherwise}
\end{cases}
$$

Full gradient inside the trust region; zero once the ratio has been pushed past the
boundary in the direction the advantage favours. This is the same three-case result as
[`../../toy_mdp/ppo.py`](../../toy_mdp/ppo.py) derives on a tabular policy — the
derivation survives the move to a language model, and autograd only carries the chain rule
the rest of the way.

Note $\rho = 1 \Rightarrow \partial\mathcal{L}/\partial\log\pi = A_t$: **with one inner
epoch the clip is inert by construction**, and this is plain actor-critic policy gradient.
The clip only starts working at `--inner-epochs 2` and above.

In [5]:
lp   = torch.randn(B, T, requires_grad=True)
old  = torch.randn(B, T)
adv  = torch.randn(B, T)
eps  = 0.2

loss, stats = ppo_actor_loss(lp, old, adv, mask, clip_eps=eps, normalize="token")
loss.backward()

# CHECK 7 -- autograd agrees with the three-case formula above, to the last bit.
with torch.no_grad():
    rho  = torch.exp(lp - old)
    hand = rho * adv
    killed = ((adv > 0) & (rho > 1 + eps)) | ((adv < 0) & (rho < 1 - eps))
    hand = torch.where(killed, torch.zeros_like(hand), hand)
    hand = -(hand * mask) / mask.sum()
assert torch.allclose(lp.grad, hand, atol=1e-6)
print(f"CHECK 7 ok  | max |autograd - hand| = {float((lp.grad-hand).abs().max()):.1e}"
      f"  (clip binding on {stats['clip_frac']:.1%} of tokens)")

# CHECK 8 -- at rho = 1 the clip is inert and the gradient is the plain PG term.
lp2 = old.clone().requires_grad_(True)
l2, s2 = ppo_actor_loss(lp2, old, adv, mask, clip_eps=eps)
l2.backward()
assert abs(s2["ratio_mean"] - 1.0) < 1e-6 and s2["clip_frac"] == 0.0
assert torch.allclose(lp2.grad, -(adv * mask) / mask.sum(), atol=1e-6)
print(f"CHECK 8 ok  | ratio_mean={s2['ratio_mean']:.6f}  clip_frac={s2['clip_frac']}")

CHECK 7 ok  | max |autograd - hand| = 0.0e+00  (clip binding on 18.8% of tokens)
CHECK 8 ok  | ratio_mean=1.000000  clip_frac=0.0


## 5. The critic loss

$$
\mathcal{L}_V = \max\Big( \big(V - G\big)^2,\ \big(V_{\text{old}} + \mathrm{clip}(V - V_{\text{old}}, \pm\epsilon_v) - G\big)^2 \Big)
$$

The clip is a trust region for the **critic**: the value may not move more than
$\epsilon_v$ from the estimate the batch was collected under. Taking the $\max$ — rather
than the $\min$ the actor takes — is what makes it a penalty rather than a licence: it can
only ever make the loss larger, never smaller.

In [6]:
v   = torch.randn(B, T, requires_grad=True)
ov  = torch.randn(B, T)
ret = torch.randn(B, T)

l_clipped, s_clipped = value_loss(v, ov, ret, mask, clip_eps=0.2)
l_plain,   _         = value_loss(v, ov, ret, mask, clip_eps=None)

# CHECK 9 -- the clip is a penalty: never below plain MSE.
assert float(l_clipped) >= float(l_plain) - 1e-6
print(f"CHECK 9 ok  | clipped {float(l_clipped):.4f} >= plain {float(l_plain):.4f}"
      f"  (binding on {s_clipped['value_clip_frac']:.1%} of tokens)")

# CHECK 10 -- whitening, when it is used at all, is over ACTIVE tokens only.
w = whiten(adv, mask)
active = mask.sum()
m = float((w * mask).sum() / active)
var = float((((w - m) * mask) ** 2).sum() / active)
assert abs(m) < 1e-5 and abs(var - 1) < 1e-4
print(f"CHECK 10 ok | whitened mean {m:.1e}, var {var:.6f} over active tokens")

CHECK 9 ok  | clipped 2.9147 >= plain 2.0842  (binding on 43.8% of tokens)
CHECK 10 ok | whitened mean 1.5e-08, var 1.000000 over active tokens


/tmp/ipykernel_431842/2104535195.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  assert float(l_clipped) >= float(l_plain) - 1e-6


### Why whitening is off by default

Whitening the advantages across a batch re-introduces a **batch-level baseline**. On a
batch of 8 *different* prompts that baseline conflates prompt difficulty with completion
quality — which is the exact job the critic was added to do. Turning it on would make the
curves look better and would make this experiment unable to answer its own question, so it
is a flag (`--whiten-advantages 1`) and not a default.

## 6. The seam GRPO never meets: left padding

A GRPO group is $G$ samples of a **single** prompt, so every sequence in the batch has the
same prompt length and there is no padding at all. This method takes one sample each from
**many** prompts, so the batch is left-padded — and a left-padded batch pushed through a
plain `model(input_ids=...)` is wrong twice over:

1. the model attends to the pad tokens, and
2. RoPE reads position $0$ at the pad rather than at the first real token.

Neither raises an error. Both quietly corrupt every log-probability in the padded rows,
which is the whole training signal. The fix is an explicit `attention_mask` plus
`position_ids = \mathrm{cumsum}(\text{mask}) - 1`, and the check below is worth its runtime:
it scores the same sequence twice, once padded and once alone.

In [7]:
from ppo_ac import Config, load_policy, forward_policy_value

cfg = Config(dtype="float32", lora_r=4)
policy, vhead, tok = load_policy(cfg, "cpu")
policy.eval()

texts = [
    "<|im_start|>user\nSay hi.<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nExplain reverse osmosis, at enough length that this prompt is "
    "clearly the longer of the two.<|im_end|>\n<|im_start|>assistant\n",
]
enc = tok(texts, return_tensors="pt", padding=True)
Ccomp = 6
seq  = torch.cat([enc["input_ids"], torch.randint(1000, 5000, (2, Ccomp))], dim=1)
attn = torch.cat([enc["attention_mask"], torch.ones(2, Ccomp, dtype=torch.long)], dim=1)

lp_batch, v_batch = forward_policy_value(policy, vhead, seq, attn, Ccomp)
print("prompt lengths", enc["attention_mask"].sum(1).tolist(),
      "| padded to", enc["input_ids"].shape[1])
print("position_ids row 0:", padded_position_ids(enc["attention_mask"])[0][:14].tolist(), "...")

/home/bayan/MembraneClaw/experiments/membrane_grpo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<00:35,  8.24it/s]

Loading weights:  63%|██████▎   | 184/290 [00:00<00:00, 992.28it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 920.59it/s]

prompt lengths [11, 29] | padded to 29
position_ids row 0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] ...


In [8]:
# CHECK 11 -- the same sequence scores the same padded and unpadded.
for i in range(2):
    keep = attn[i].bool()
    lone_seq = seq[i][keep].unsqueeze(0)
    lp_alone, v_alone = forward_policy_value(
        policy, vhead, lone_seq, torch.ones_like(lone_seq), Ccomp
    )
    d = float((lp_alone[0] - lp_batch[i]).abs().max())
    assert d < 2e-3, f"row {i} differs by {d}"
    print(f"CHECK 11 ok | row {i}: max |dlogp| padded vs alone = {d:.2e}")

# The control: what the naive forward would have produced.
naive = policy(input_ids=seq, logits_to_keep=Ccomp + 1, output_hidden_states=True)
lp_naive = ppo_ac.selective_logprobs(naive.logits[:, :-1], seq[:, -Ccomp:])
print("\nwithout the fix, the same rows are off by:")
for i in range(2):
    print(f"  row {i} ({int(enc['attention_mask'][i].sum())} real prompt tokens):"
          f" {float((lp_naive[i]-lp_batch[i]).abs().max()):.3f} nats")

CHECK 11 ok | row 0: max |dlogp| padded vs alone = 7.63e-06


CHECK 11 ok | row 1: max |dlogp| padded vs alone = 0.00e+00



without the fix, the same rows are off by:
  row 0 (11 real prompt tokens): 9.819 nats
  row 1 (29 real prompt tokens): 0.000 nats


The short row — the one that actually gets padded — is off by around **4 nats per token**.
The long row, which needs no padding, is exact. That is the signature of this class of bug:
it is invisible in any test where the batch happens to be uniform, and GRPO's batches
always are.

## 7. The critic's learning rate is a derived quantity

A linear head trained with Adam moves each of its $d$ weights by about `lr` per step, and
those steps are summed through the features:

$$
|\Delta V| \;\approx\; \text{lr} \cdot \lVert h \rVert_1 .
$$

$\lVert h \rVert_1$ is measured below, not assumed. The reward lives in $[0, 1]$, so a
sane critic step is on the order of $0.02$ — which pins `lr` two to three orders of
magnitude below anything that would look reasonable by eye.

**This is not hypothetical.** The first CPU smoke run of this file used `value_lr=1e-3` and
the critic went $0.086 \to -1.868$ in a single step, driving every advantage positive and
the gradient norm from 10 to 239. The run below is what caught it.

In [9]:
from ppo_ac import load_cases, build_messages

case = load_cases("train")[0]
text = tok.apply_chat_template(build_messages(case["record"]), tokenize=False,
                               add_generation_prompt=True)
enc1 = tok([text], return_tensors="pt")
Cc = 8
s1 = torch.cat([enc1["input_ids"], torch.randint(1000, 5000, (1, Cc))], dim=1)
a1 = torch.ones_like(s1)
out = policy(input_ids=s1, attention_mask=a1, logits_to_keep=Cc + 1, output_hidden_states=True)
h = out.hidden_states[-1][:, -(Cc + 1):-1].detach()

l1 = float(h.abs().sum(-1).mean())
print(f"hidden size {h.shape[-1]} | E|h| = {float(h.abs().mean()):.3f} | ||h||_1 = {l1:.0f}\n")
for lr in (1e-3, 1e-4, 1e-5, 5e-6):
    flag = "  <-- the default" if lr == 5e-6 else ("   (what broke it)" if lr == 1e-3 else "")
    print(f"  value_lr {lr:.0e}  ->  |dV| per step ~ {l1 * lr:7.4f}{flag}")

# CHECK 12 -- the shipped default lands the critic step inside the reward's own scale.
assert l1 * 5e-6 < 0.05, "value_lr default is too large for this model's hidden scale"
print(f"\nCHECK 12 ok | default step {l1 * 5e-6:.4f} << reward range 1.0")

hidden size 896 | E|h| = 4.824 | ||h||_1 = 4323

  value_lr 1e-03  ->  |dV| per step ~  4.3226   (what broke it)
  value_lr 1e-04  ->  |dV| per step ~  0.4323
  value_lr 1e-05  ->  |dV| per step ~  0.0432
  value_lr 5e-06  ->  |dV| per step ~  0.0216  <-- the default

CHECK 12 ok | default step 0.0216 << reward range 1.0


## 8. The pathology: a non-negative reward and a cold critic

The reward is non-negative. A value head initialised to **zero** therefore yields
$A_t = R \ge 0$ for every token of every completion, and the first updates reinforce
*everything* — garbage included. GRPO cannot do this: group centring makes advantages
zero-mean by construction.

The default initialises the head's bias to $0.086$, the frozen 0.5B baseline's mean reward
from `runs/baseline-0.5b-v2/eval_dev_greedy.json`. Not a tuned constant — a measurement
this repository already owns.

In [10]:
R_batch = torch.tensor([0.0, 0.05, 0.10, 0.25])
rew_b = terminal_rewards(R_batch, mask)

cold = ValueHead(8, init_bias=0.0)
warm = ValueHead(8, init_bias=VALUE_INIT_BIAS)
feats = torch.randn(B, T, 8)

A_cold, _ = gae(rew_b, cold(feats).detach() * mask, mask, lam=1.0)
A_warm, _ = gae(rew_b, warm(feats).detach() * mask, mask, lam=1.0)

frac_neg = lambda A: float(((A < 0) & (mask > 0)).float().sum() / mask.sum())

# CHECK 13 -- a zero-init critic can never produce a NEGATIVE advantage, because the
# reward is non-negative and V is 0: the update has no way to push anything down. The
# calibrated critic splits the batch, which is the whole function of a baseline.
assert frac_neg(A_cold) == 0.0
assert frac_neg(A_warm) > 0.0
print(f"CHECK 13 ok | zero-init critic:  {frac_neg(A_cold):.0%} of advantages negative "
      f"-> nothing can be pushed down; every completion is reinforced")
print(f"            | bias={VALUE_INIT_BIAS} critic: {frac_neg(A_warm):.0%} of advantages negative "
      f"-> below-baseline completions are pushed down")
print(f"\n            | rewards in this batch: {R_batch.tolist()}  (mean {float(R_batch.mean()):.3f})")

CHECK 13 ok | zero-init critic:  0% of advantages negative -> nothing can be pushed down; every completion is reinforced
            | bias=0.086 critic: 75% of advantages negative -> below-baseline completions are pushed down

            | rewards in this batch: [0.0, 0.05000000074505806, 0.10000000149011612, 0.25]  (mean 0.100)


## 9. The point of the whole exercise

GRPO's reference CPU smoke run hit a degenerate group on step 2 — four completions all
scoring exactly $0.0$, within-group variance gone, **gradient norm 0.00000**. That is not
an edge case: 16% of groups are degenerate at the frozen baseline.

A critic has no such hole. When every reward in the batch is identical, $R - V$ is still
non-zero, and the update still says something — *"all of these were worse than expected,
make them less likely"*. Which is true, and which GRPO structurally cannot say.

In [11]:
# CHECK 14 -- identical rewards: GRPO's advantages are exactly zero, this one's are not.
from grpo_scratch import group_advantages

identical = [0.0, 0.0, 0.0, 0.0]
grpo_adv, degenerate = group_advantages(identical)
assert degenerate and all(a == 0.0 for a in grpo_adv)

V = torch.full((B, T), 0.086)
A_ac, _ = gae(terminal_rewards(torch.tensor(identical), mask), V, mask, lam=1.0)
assert float((A_ac * mask).abs().sum()) > 0

print(f"CHECK 14 ok | same four rewards {identical}")
print(f"            | GRPO advantages   = {grpo_adv}  (degenerate={degenerate}, no gradient)")
print(f"            | actor-critic A_t  = {float(A_ac[0,0]):.3f} on every active token")
print(f"            | -> a gradient exists, and it points the right way")

CHECK 14 ok | same four rewards [0.0, 0.0, 0.0, 0.0]
            | GRPO advantages   = [0.0, 0.0, 0.0, 0.0]  (degenerate=True, no gradient)
            | actor-critic A_t  = -0.086 on every active token
            | -> a gradient exists, and it points the right way


In [12]:
run = HERE / "runs" / "smoke-ppo-ac-cpu" / "metrics.jsonl"
if run.exists():
    rows = [json.loads(l) for l in run.read_text().splitlines()]
    print("The committed CPU smoke run of ppo_ac.py, for comparison with GRPO's:\n")
    print(f"{'step':>4} {'reward':>8} {'V':>8} {'dV':>8} {'adv_mean':>9} {'|grad|':>8}")
    for r in rows:
        print(f"{r['step']:>4} {r['reward_mean']:>8.4f} {r['value_mean']:>8.4f} "
              f"{r['value_step']:>8.4f} {r['adv_mean']:>+9.4f} {r['grad_norm']:>8.3f}")
    z = [r for r in rows if r["reward_mean"] == 0.0]
    if z:
        s = z[0]
        print(f"\nStep {s['step']} is the degenerate case in the wild: every reward exactly 0.0.")
        print(f"GRPO's own smoke run hit this on step 2 and recorded |grad| = 0.00000.")
        print(f"Here |grad| = {s['grad_norm']:.3f}.")
else:
    print("no committed CPU run found; see 02 for the GPU run")

The committed CPU smoke run of ppo_ac.py, for comparison with GRPO's:

step   reward        V       dV  adv_mean   |grad|
   0   0.0400   0.0860   0.0000   -0.0060    5.495
   1   0.0200   0.0767   0.0093   -0.0367    7.334
   2   0.0000   0.0686   0.0081   -0.0686    7.806
   3   0.0500   0.0585   0.0101   +0.0115    0.931

Step 2 is the degenerate case in the wild: every reward exactly 0.0.
GRPO's own smoke run hit this on step 2 and recorded |grad| = 0.00000.
Here |grad| = 7.806.


## What this notebook did not check

* **That any of it learns.** 14 checks say the mathematics is implemented correctly and the
  plumbing is not silently corrupting the signal. None of them say the reward will rise.
  That is [`02_ppo_actor_critic_0.5b.ipynb`](02_ppo_actor_critic_0.5b.ipynb), and it needs
  the GPU.
* **The critic's accuracy.** A linear probe on a 0.5B's hidden states may simply not be
  able to predict this reward. If `value_mae` stays flat near the reward's own standard
  deviation, the critic is predicting the mean and this is REINFORCE with an expensive
  constant baseline — a real possible outcome, and one worth reporting rather than hiding.
* **Whether per-token credit assignment helps.** $\lambda$ is the dial; $\lambda = 1$ is
  the honest default and the ablation is in 02.